# 02_text_preprocessing: Stemming vs Lemmatization Latency, BPE, and WordPiece Simulations

This notebook implements comparative benchmarks for stemming vs. lemmatization using NLTK on Gutenberg corpus text. It also implements programmatic simulations of Byte-Pair Encoding (BPE) vocabulary merges and WordPiece co-occurrence scoring ratios.

In [1]:
%matplotlib inline
import time
import nltk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Load Gutenberg sentences
nltk.download('gutenberg', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import gutenberg

words = [w.lower() for w in gutenberg.words('carroll-alice.txt')[:1000] if w.isalpha()]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Benchmark Stemmer
start_time = time.perf_counter()
for w in words:
    stemmer.stem(w)
stem_time = (time.perf_counter() - start_time) * 1000 # in ms

# Benchmark Lemmatizer
start_time = time.perf_counter()
for w in words:
    lemmatizer.lemmatize(w, pos='v')
lemma_time = (time.perf_counter() - start_time) * 1000 # in ms

print(f"Stemmer Latency for 1,000 words: {stem_time:.2f} ms")
print(f"Lemmatizer Latency for 1,000 words: {lemma_time:.2f} ms")

# Plot results
fig, ax = plt.subplots(figsize=(5, 3), dpi=150)
bars = ax.bar(['Porter Stemmer', 'WordNet Lemmatizer'], [stem_time, lemma_time], color=['#3b82f6', '#8b5cf6'], width=0.4)
ax.set_ylabel('Latency (ms)')
ax.set_title('Lexical Reduction Latency for 1,000 Words')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval:.1f}ms", ha='center', va='bottom', fontsize=8, weight='bold')
plt.tight_layout()
plt.show()

assert stem_time < lemma_time, "Heuristic stemmer should be faster than dictionary lemmatizer!"

Stemmer Latency for 1,000 words: 4.34 ms
Lemmatizer Latency for 1,000 words: 1524.66 ms


C:\Users\aryan\AppData\Local\Temp\ipykernel_28572\3816667187.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Output Explanation: Latency Benchmark
- **Speed Difference:** The Porter Stemmer runs significantly faster than the WordNet Lemmatizer. This is because stemming uses a simple, rule-based suffix-chopping heuristic (e.g., regex checks on string length) that executes in linear time without memory overhead.
- **Lemmatization Overhead:** The lemmatizer is slower because it consults a dictionary database (WordNet), parsing morphological dependencies and grammatical context. 
- **Production Trade-off:** Use stemming when throughput is the main priority (e.g., streaming logs classifications). Use lemmatization when semantic correctness is critical (e.g., dictionary mapping, text generation).

In [2]:
import re
from collections import defaultdict

# BPE counts helper
def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

# BPE merge helper
def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Initialize counts matching study guide
vocab = {
    "h u g _": 10,
    "p u g _": 5,
    "h u g s _": 5
}

print("Initial Corpus Vocabulary:")
print(vocab)

# Iteration 1
pairs = get_stats(vocab)
best_pair = max(pairs, key=pairs.get)
print(f"\nIteration 1 - Most frequent pair: {best_pair} ({pairs[best_pair]} occurrences)")
assert best_pair == ("u", "g"), "Iteration 1 best pair mismatch!"
vocab = merge_vocab(best_pair, vocab)
print("Corpus after Merge 1:")
print(vocab)

# Iteration 2
pairs = get_stats(vocab)
best_pair = max(pairs, key=pairs.get)
print(f"\nIteration 2 - Most frequent pair: {best_pair} ({pairs[best_pair]} occurrences)")
assert best_pair == ("h", "ug"), "Iteration 2 best pair mismatch!"
vocab = merge_vocab(best_pair, vocab)
print("Corpus after Merge 2:")
print(vocab)

Initial Corpus Vocabulary:
{'h u g _': 10, 'p u g _': 5, 'h u g s _': 5}

Iteration 1 - Most frequent pair: ('u', 'g') (20 occurrences)
Corpus after Merge 1:
{'h ug _': 10, 'p ug _': 5, 'h ug s _': 5}

Iteration 2 - Most frequent pair: ('h', 'ug') (15 occurrences)
Corpus after Merge 2:
{'hug _': 10, 'p ug _': 5, 'hug s _': 5}


### Output Explanation: BPE Merges
- **Merge 1:** The pair `('u', 'g')` co-occurs $10 + 5 + 5 = 20$ times in the corpus, making it the most frequent pair. It merges to form the subword `ug`.
- **Merge 2:** The pair `('h', 'ug')` co-occurs $10 + 5 = 15$ times, tying with `('ug', '_')`. Word boundary heuristics choose `('h', 'ug')`, merging them to form the subword `hug`.
- **Consistency:** The output logs match the manual hand-calculations in Module 02, verifying the mathematical correctness of BPE's bottom-up consolidation.

In [3]:
# Mock counts matching study guide
N = 100 # Total corpus count

count_h = 20
count_u = 30
count_hu = 15

count_p = 5
count_pu = 4

# Calculate WordPiece scores
score_hu = count_hu / (count_h * count_u)
score_pu = count_pu / (count_p * count_u)

print(f"WordPiece Score for ('h', 'u'): {score_hu:.4f}")
print(f"WordPiece Score for ('p', 'u'): {score_pu:.4f}")

# Verification assertion
assert score_pu > score_hu, "WordPiece scoring logic assertion failed!"
print("\nSuccess: score_pu > score_hu, meaning ('p', 'u') is merged first despite lower absolute co-occurrence count.")

WordPiece Score for ('h', 'u'): 0.0250
WordPiece Score for ('p', 'u'): 0.0267

Success: score_pu > score_hu, meaning ('p', 'u') is merged first despite lower absolute co-occurrence count.


### Output Explanation: WordPiece Scoring
- **Score Comparison:** The score of the pair `('p', 'u')` ($0.0267$) is higher than `('h', 'u')` ($0.0250$), even though `('h', 'u')` appears far more times in absolute counts ($15$ vs $4$).
- **Statistical Motivation:** WordPiece divides the co-occurrence count by the independent count product. This normalizes the score against raw character frequency, prioritizing pairs that have a high statistical correlation over common letters that happen to appear together frequently by random chance.